In [ ]:
#| default_exp core.precision

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import warnings
from dataclasses import asdict, dataclass

## Overview

Compression asks four questions. *What* block of parameters to remove is [granularity](granularity.html),
*which* ones matter is [criteria](criteria.html), *when* to act is [schedules](schedules.html) — and *how
few bits* to keep the survivors in is **precision**.

A precision request has five parts:

| Part | Argument | Values |
|---|---|---|
| Weight width | `weight_bits` | `4`, `8`, `16`, or a `{layer_name: width}` dict |
| Activation width | `act_bits` | `8`, `16` |
| Weight axis | `qscheme` | `'per_tensor'`, `'per_channel'`, `'per_group'` |
| Symmetry | `symmetric` | `True` (every zero-point is 0), `False` (affine) |
| Q/DQ placement | `qdq_placement` | `'per_op'`, `'skip_conv_add'` (the `pt2e` cell only) |

Two conventions run through the grammar:

- **A width of 16 means *not quantized***: the tensor keeps the model's floating-point dtype. One label
  then covers fp16 and fp32 models, so INT8 weight-only quantization is written `W8A16` whatever the
  model's float type. `weight_bits={'head': 16}` is how you keep one layer out of the quantization.
- **The finer the axis, the more scales**: `per_tensor` keeps one scale for the whole tensor,
  `per_channel` one per output channel, `per_group` one per fixed-size block of weights.
- **A placement is about *where the pairs sit*, not how many bits**: `per_op` quantizes the result of
  every operator the flow annotates; `skip_conv_add` leaves one edge — the residual branch's
  convolution feeding its add — without a Q/DQ pair, so that result reaches the add at accumulator
  precision. It changes the arithmetic, so it is opt-in and recorded on the spec.

Not every combination exists. A backend can only apply the cells its observers and kernels implement, and
only some of those survive an ONNX export. `PRECISION_SUPPORT` is that matrix, and `Quantizer` validates
every request against it **at construction**: a cell a backend cannot honor raises immediately, naming a
backend that can, instead of being silently rounded to something the backend *can* do.

## Precision cells

A `PrecisionCell` is one row of the matrix: a backend, a precision, and what that pair can do.

In [ ]:
#| export
QSCHEMES = ('per_tensor', 'per_channel', 'per_group')  # the weight axes this grammar names
WIDTHS = (4, 8, 16)                                    # the bit widths it names; 16 = left in floating point
QDQ_PLACEMENTS = ('per_op', 'skip_conv_add')  # where a flow may put its Q/DQ pairs; the first one quantizes every operator it annotates


def _label(weight_bits: int, act_bits: int) -> str:
    "Short name of a precision, e.g. 'W8A8'"
    return f"W{weight_bits}A{act_bits}"


@dataclass(frozen=True, slots=True)
class PrecisionCell:
    "What one (backend, weight width, activation width) combination can do"
    backend: str               # `Quantizer` backend
    weight_bits: int           # weight width
    act_bits: int              # activation width (16: activations stay in floating point)
    qschemes: tuple[str, ...]  # weight axes the cell can honor; the first one is its default
    symmetries: tuple[bool, ...]  # symmetry settings it can honor; the first one is its default
    per_layer: bool            # honors a {layer_name: width} dict
    exports: bool              # `export_qdq` can write it as a QDQ ONNX graph
    note: str                  # what it does, and why it does not export when it does not
    default_group_size: int | None = None  # group size the backend picks when the axis is 'per_group'
    qdq_placements: tuple[str, ...] = ()  # Q/DQ placements it can honor, first one its default; empty = the cell has no such axis

    @property
    def label(self) -> str:
        "Short name of the precision, e.g. 'W8A8'"
        return _label(self.weight_bits, self.act_bits)

    @property
    def default_qscheme(self) -> str:
        "Weight axis used when the caller does not name one"
        return self.qschemes[0]

    @property
    def default_symmetric(self) -> bool:
        "Symmetry used when the caller does not name one"
        return self.symmetries[0]

    @property
    def default_qdq_placement(self) -> str | None:
        "Q/DQ placement used when the caller does not name one, or None when the cell has no such axis"
        return self.qdq_placements[0] if self.qdq_placements else None

    def as_dict(self) -> dict:
        "Plain-dict view, for logging or serialization"
        return asdict(self)


_AFFINE_FX = ("FX observers keep activations affine (unsigned, non-zero zero-point), so the graph is not "
              "a portable Q/DQ export.")

_CELLS = (
    PrecisionCell('pt2e', 8, 8, ('per_channel', 'per_tensor'), (True, False), False, True,
                  "Symmetric INT8 weights and activations: every zero-point is 0, which is what `export_qdq` "
                  "writes into a QDQ ONNX graph. The one cell that also chooses where its Q/DQ pairs sit.",
                  qdq_placements=('per_op', 'skip_conv_add')),
    PrecisionCell('x86', 8, 8, ('per_channel', 'per_tensor'), (False,), True, False, _AFFINE_FX),
    PrecisionCell('fbgemm', 8, 8, ('per_channel', 'per_tensor'), (False,), True, False, _AFFINE_FX),
    PrecisionCell('onednn', 8, 8, ('per_channel', 'per_tensor'), (False,), True, False, _AFFINE_FX),
    PrecisionCell('qnnpack', 8, 8, ('per_tensor',), (False,), True, False, _AFFINE_FX),
    PrecisionCell('torchao', 8, 8, ('per_channel',), (True,), False, False,
                  "INT8 weights with dynamically quantized activations: the activation scales are computed at "
                  "run time, so they cannot be written into a static Q/DQ graph."),
    PrecisionCell('torchao', 8, 16, ('per_channel', 'per_group'), (True,), True, False,
                  "INT8 weight-only: the activations stay in floating point, so there is no activation Q/DQ "
                  "pair to export. This is the one torchao cell that honors a per-layer `weight_bits` "
                  "dict, over the Linear layers torchao rewrites."),
    PrecisionCell('torchao', 4, 16, ('per_group',), (True,), False, False,
                  "INT4 weight-only, a size lever: only torchao ships the kernels, and ONNX opset 18 has no "
                  "INT4 Q/DQ pair.", default_group_size=128),
)

PRECISION_SUPPORT: dict[tuple[str, int, int], PrecisionCell] = {
    (c.backend, c.weight_bits, c.act_bits): c for c in _CELLS}

_BACKENDS = tuple(dict.fromkeys(c.backend for c in _CELLS))
_LEGACY_BACKENDS = ('x86', 'fbgemm', 'onednn', 'qnnpack')


def _backends_where(predicate) -> list[str]:
    "Backends with at least one cell satisfying `predicate` — how a refusal names who CAN honor a request"
    return sorted({c.backend for c in _CELLS if predicate(c)})


def precision_table() -> str:
    "Markdown view of `PRECISION_SUPPORT` — the matrix is the code, this is only its rendering"
    head = ("| Backend | Precision | Weight axis | Symmetric | Per-layer | Q/DQ placement | "
            "`export_qdq` | Notes |\n"
            "|---|---|---|---|---|---|---|---|\n")
    rows = [f"| `{c.backend}` | {c.label} | {', '.join(c.qschemes)} | "
            f"{', '.join(str(s) for s in c.symmetries)} | {'yes' if c.per_layer else 'no'} | "
            f"{', '.join(c.qdq_placements) or 'n/a'} | "
            f"{'yes' if c.exports else 'no'} | {c.note} |" for c in _CELLS]
    return head + "\n".join(rows)

In [ ]:
show_doc(PrecisionCell)

## The support matrix

The table below is **rendered from `PRECISION_SUPPORT`** rather than written by hand, so it cannot drift
away from what the code enforces.

In [ ]:
from IPython.display import Markdown
Markdown(precision_table())

In [ ]:
show_doc(precision_table)

## The resolved spec

`_resolve_spec` turns a request into exactly one `QuantSpec`, or raises. `Quantizer` attaches the result
to the model it quantizes, where `quant_spec(model)` reads it back — that is what `export_qdq` consults
to refuse a precision it cannot write.

In [ ]:
#| export
SPEC_ATTR = '_fasterai_quant_spec'  # attribute `Quantizer` leaves on the models it quantizes


@dataclass(frozen=True, slots=True)
class QuantSpec:
    "The single precision cell a `Quantizer` resolved to — attached to the model it quantizes"
    backend: str                        # backend that will apply it
    method: str                         # 'static', 'dynamic', 'qat' or a torchao recipe
    weight_bits: int                    # weight width
    act_bits: int                       # activation width
    qscheme: str                        # weight axis
    symmetric: bool                     # True when every zero-point is 0
    group_size: int | None = None       # weights sharing one scale (qscheme='per_group')
    layer_bits: dict | None = None      # per-layer widths, when the caller asked for some
    qdq_placement: str | None = None    # where the Q/DQ pairs sit; None on a backend with no such axis

    @property
    def label(self) -> str:
        "Short name of the precision, e.g. 'W8A8'"
        return _label(self.weight_bits, self.act_bits)

    @property
    def cell(self) -> PrecisionCell:
        "Row of `PRECISION_SUPPORT` this spec was validated against"
        return PRECISION_SUPPORT[(self.backend, self.weight_bits, self.act_bits)]

    @property
    def exports(self) -> bool:
        "Whether `export_qdq` can write this precision as a QDQ ONNX graph"
        return self.cell.exports

    @property
    def note(self) -> str:
        "What the cell does, and why it does not export when it does not"
        return self.cell.note

    def as_dict(self) -> dict:
        "Plain-dict view, for logging or serialization"
        return asdict(self)


def quant_spec(
    model,  # Any model, quantized or not
) -> QuantSpec | None:
    "The precision `Quantizer` applied to `model`, or `None` when fasterai did not quantize it"
    return getattr(model, SPEC_ATTR, None)

In [ ]:
show_doc(QuantSpec)

In [ ]:
show_doc(quant_spec)

## Resolution

One resolver, one place where a request is accepted or refused. Every refusal names the argument at fault
and, when there is one, a backend that can honor it.

In [ ]:
#| export
_TORCHAO_CELL = {'int8_weight_only': (8, 16), 'int8_dynamic': (8, 8), 'int4_weight_only': (4, 16)}
_TORCHAO_RECIPES = {_label(w, a): method for method, (w, a) in _TORCHAO_CELL.items()}


def _type_error(name: str, expectation: str, value) -> TypeError:
    "One shape for every type complaint, so they all read the same"
    return TypeError(f"`{name}` must be {expectation}, got {value!r} ({type(value).__name__}).")


def _check_width(name: str, value) -> int:
    "Validate one bit width, naming the argument that carried it"
    if isinstance(value, bool) or not isinstance(value, int):
        raise _type_error(name, f"an int, one of {list(WIDTHS)}", value)
    if value not in WIDTHS:
        raise ValueError(f"`{name}={value}` is not a width this grammar names. Use one of {list(WIDTHS)} "
                         "(16 means 'left in floating point').")
    return value


def _split_weight_bits(weight_bits) -> tuple[int | None, dict | None]:
    "Split `weight_bits` into a uniform width and the per-layer widths it may carry"
    if weight_bits is None: return None, None
    if isinstance(weight_bits, dict):
        if not weight_bits:
            raise ValueError("`weight_bits={}` names no layer. Pass an int for a uniform width, or a "
                             "{layer_name: width} dict with at least one entry.")
        for name, bits in weight_bits.items():
            if not isinstance(name, str):
                raise _type_error('weight_bits keys', 'layer names (str)', name)
            if _check_width(f"weight_bits['{name}']", bits) not in (8, 16):
                raise ValueError(f"`weight_bits['{name}']={bits}`: a per-layer width is 8 (quantize this "
                                 "layer) or 16 (leave it in floating point).")
        # This scalar does not set the width — `_resolve_method` does. It steers which precision CELL
        # the request lands in, and therefore which refusal it gets: a dict holding an 8 is a request
        # for a W8 cell, while a dict that only says 16 asks for no quantized layer at all and leaves
        # the cell to the backend, which is how `weight_bits={'fc': 16}` has always resolved.
        return (8 if 8 in weight_bits.values() else None), dict(weight_bits)
    return _check_width('weight_bits', weight_bits), None


def _resolve_method(backend: str, method: str, weight_bits: int | None,
                    act_bits: int | None) -> tuple[str, int, int]:
    "Reconcile `method` with the requested widths, and let the widths pick a torchao recipe"
    if backend != 'torchao':
        return method, weight_bits if weight_bits is not None else 8, act_bits if act_bits is not None else 8
    if method in _TORCHAO_CELL:
        w, a = _TORCHAO_CELL[method]
        for name, asked, native in (('weight_bits', weight_bits, w), ('act_bits', act_bits, a)):
            if asked is not None and asked != native:
                raise ValueError(
                    f"backend='torchao' method='{method}' is a {_label(w, a)} recipe, which "
                    f"`{name}={asked}` contradicts. Drop `{name}`, or name the recipe that matches: "
                    f"{_TORCHAO_RECIPES}.")
        return method, w, a
    if weight_bits is None and act_bits is None:
        raise ValueError(f"backend='torchao' has no method '{method}'. Its recipes are "
                         f"{sorted(_TORCHAO_CELL)}, or name the precision directly with "
                         "`weight_bits=` / `act_bits=`.")
    w = weight_bits if weight_bits is not None else 8
    a = act_bits if act_bits is not None else (16 if w == 4 else 8)
    derived = _TORCHAO_RECIPES.get(_label(w, a))
    if derived is None:
        raise ValueError(f"backend='torchao' has no recipe for {_label(w, a)}. It ships "
                         f"{_TORCHAO_RECIPES}.")
    return derived, w, a


def _lookup_cell(backend: str, weight_bits: int, act_bits: int) -> PrecisionCell:
    "Row of `PRECISION_SUPPORT` for this precision, or the reason there is none"
    cell = PRECISION_SUPPORT.get((backend, weight_bits, act_bits))
    if cell is not None: return cell
    label = _label(weight_bits, act_bits)
    if backend not in _BACKENDS:
        raise ValueError(f"Unknown backend '{backend}'. fasterai quantizes with {list(_BACKENDS)}.")
    here = [c.label for c in _CELLS if c.backend == backend]
    elsewhere = _backends_where(lambda c: (c.weight_bits, c.act_bits) == (weight_bits, act_bits))
    if elsewhere:
        raise ValueError(f"backend='{backend}' cannot quantize {label}; it runs {here}. "
                         f"The backend(s) that run {label}: {elsewhere}.")
    raise ValueError(f"No fasterai backend runs {label}: nothing ships a kernel for it. "
                     f"backend='{backend}' runs {here}.")


def _resolve_qscheme(cell: PrecisionCell, qscheme, group_size, use_per_tensor: bool) -> str:
    "Pick the weight axis, refusing any the cell cannot honor"
    if qscheme is not None:
        if not isinstance(qscheme, str):
            raise _type_error('qscheme', f"one of {list(QSCHEMES)} (str)", qscheme)
        if qscheme not in QSCHEMES:
            raise ValueError(f"Unknown qscheme '{qscheme}'. The weight axes this grammar names are "
                             f"{list(QSCHEMES)}.")
    if use_per_tensor:
        if cell.backend not in _LEGACY_BACKENDS:
            raise ValueError(f"`use_per_tensor=True` is a legacy-backend flag that backend='{cell.backend}' "
                             "never read. Ask for the axis instead: qscheme='per_tensor'.")
        if qscheme not in (None, 'per_tensor'):
            raise ValueError(f"`use_per_tensor=True` and `qscheme='{qscheme}'` ask for different weight "
                             "axes. Keep one.")
        qscheme = 'per_tensor'
    if qscheme is None:
        qscheme = 'per_group' if (group_size is not None and 'per_group' in cell.qschemes) \
            else cell.default_qscheme
    if qscheme not in cell.qschemes:
        able = _backends_where(lambda c: qscheme in c.qschemes)
        raise ValueError(f"backend='{cell.backend}' {cell.label} quantizes weights {list(cell.qschemes)}, "
                         f"not '{qscheme}'. The backend(s) that do: {able}.")
    return qscheme


def _resolve_group_size(cell: PrecisionCell, qscheme: str, group_size) -> int | None:
    "Validate the group size against the axis that gives it a meaning"
    if group_size is None:
        if qscheme != 'per_group': return None
        if cell.default_group_size is None:
            raise ValueError("qscheme='per_group' needs a `group_size` (e.g. group_size=128): a group is a "
                             "fixed number of weights sharing one scale.")
        return cell.default_group_size
    if isinstance(group_size, bool) or not isinstance(group_size, int):
        raise _type_error('group_size', 'a positive int', group_size)
    if group_size <= 0:
        raise ValueError(f"`group_size={group_size}` is not a size: a group holds at least one weight.")
    if qscheme != 'per_group':
        able = _backends_where(lambda c: 'per_group' in c.qschemes)
        raise ValueError(f"`group_size={group_size}` only means something with qscheme='per_group'; "
                         f"backend='{cell.backend}' {cell.label} quantizes weights '{qscheme}'. "
                         f"The backend(s) that quantize per group: {able}.")
    return group_size


def _resolve_qdq_placement(cell: PrecisionCell, qdq_placement) -> str | None:
    "Pick where the Q/DQ pairs sit, refusing a placement the cell's flow cannot produce"
    # Like `group_size`, this axis is left at None on a cell that has none, rather than recorded as a
    # default no flow would read: `qdq_placement` describes the arithmetic, so it may only say
    # something on the backend that can actually place the pairs.
    if qdq_placement is None: return cell.default_qdq_placement
    if not isinstance(qdq_placement, str):
        raise _type_error('qdq_placement', f"one of {list(QDQ_PLACEMENTS)} (str)", qdq_placement)
    if qdq_placement not in QDQ_PLACEMENTS:
        raise ValueError(f"Unknown qdq_placement '{qdq_placement}'. The Q/DQ placements this grammar "
                         f"names are {list(QDQ_PLACEMENTS)}.")
    if qdq_placement not in cell.qdq_placements:
        able = _backends_where(lambda c: qdq_placement in c.qdq_placements)
        # The first arm is for the cell that names SOME placements but not this one. No cell is in
        # that state today (pt2e names both), and it is kept so that adding one cannot silently ship
        # the wrong explanation — the same reason `_resolve_qscheme` names the axes a cell does have.
        why = (f"it places them {list(cell.qdq_placements)}" if cell.qdq_placements else
               "its flow quantizes every operator it rewrites and names no placement at all")
        raise ValueError(f"backend='{cell.backend}' {cell.label} cannot honor "
                         f"qdq_placement='{qdq_placement}' — {why}. The backend(s) that can: {able}.")
    return qdq_placement


def _resolve_symmetry(cell: PrecisionCell, symmetric) -> bool:
    "Pick the symmetry, refusing the one the cell's observers cannot produce"
    if symmetric is None: return cell.default_symmetric
    if not isinstance(symmetric, bool):
        raise _type_error('symmetric', 'True, False or None', symmetric)
    if symmetric not in cell.symmetries:
        able = _backends_where(lambda c: symmetric in c.symmetries)
        why = ("its observers are affine by construction: activations carry a non-zero zero-point"
               if symmetric else "it quantizes symmetrically by construction")
        raise ValueError(f"backend='{cell.backend}' {cell.label} cannot honor symmetric={symmetric} — "
                         f"{why}. The backend(s) that can: {able}.")
    if not symmetric and cell.exports:
        warnings.warn("symmetric=False keeps the activations affine: the exported graph then carries "
                      "non-zero zero-points, which some runtimes refuse.", UserWarning, stacklevel=2)
    return symmetric


def _resolve_spec(
    backend: str = 'x86',            # Target backend
    method: str = 'static',          # Quantization method, or a torchao recipe
    *,
    weight_bits=None,                # Weight width, or a {layer_name: width} dict; None = the backend's native width
    act_bits=None,                   # Activation width; None = the backend's native width
    qscheme: str | None = None,      # Weight axis; None = the backend's default
    group_size: int | None = None,   # Weights sharing one scale (qscheme='per_group')
    symmetric: bool | None = None,   # Force zero-point 0; None = the backend's default
    qdq_placement: str | None = None,  # Where the Q/DQ pairs sit: 'per_op', 'skip_conv_add'; None = the backend's default
    use_per_tensor: bool = False,    # Legacy per-tensor flag, kept in sync with `qscheme`
) -> QuantSpec:
    "Resolve the precision grammar into the one `QuantSpec` a backend will apply, or say why it cannot"
    if not isinstance(backend, str):
        raise _type_error('backend', 'a str', backend)
    weight_bits, layer_bits = _split_weight_bits(weight_bits)
    if act_bits is not None: _check_width('act_bits', act_bits)
    method, weight_bits, act_bits = _resolve_method(backend, method, weight_bits, act_bits)
    cell = _lookup_cell(backend, weight_bits, act_bits)
    qscheme = _resolve_qscheme(cell, qscheme, group_size, use_per_tensor)
    group_size = _resolve_group_size(cell, qscheme, group_size)
    symmetric = _resolve_symmetry(cell, symmetric)
    qdq_placement = _resolve_qdq_placement(cell, qdq_placement)
    if layer_bits and not cell.per_layer:
        # A backend that honors per-layer widths in ANOTHER precision is the common near-miss: name that
        # precision and the argument that reaches it, rather than sending the caller to another backend.
        sibling = next((c for c in _CELLS if c.backend == backend and c.per_layer), None)
        if sibling is not None:
            raise ValueError(f"backend='{backend}' cannot honor a per-layer `weight_bits` dict at "
                             f"{cell.label}, only at {sibling.label}: add act_bits={sibling.act_bits}.")
        raise ValueError(f"backend='{backend}' quantizes the whole model at once: it cannot honor a "
                         f"per-layer `weight_bits` dict. The backend(s) that can: "
                         f"{_backends_where(lambda c: c.per_layer)}.")
    return QuantSpec(backend=backend, method=method, weight_bits=weight_bits, act_bits=act_bits,
                     qscheme=qscheme, symmetric=symmetric, group_size=group_size, layer_bits=layer_bits,
                     qdq_placement=qdq_placement)

---

## Usage Examples

The grammar is reached through `Quantizer`, which forwards its precision arguments to `_resolve_spec`:

```python
from fasterai.quantize.quantizer import Quantizer
from fasterai.core.all import quant_spec

# The deployable cell: symmetric INT8, per-channel weights, exportable as QDQ ONNX
quantizer = Quantizer(backend='pt2e', weight_bits=8, act_bits=8, qscheme='per_channel', symmetric=True)
model_q = quantizer.quantize(model, calibration_dl=dls.valid)
print(quant_spec(model_q).as_dict())
# {'backend': 'pt2e', 'method': 'static', 'weight_bits': 8, 'act_bits': 8, 'qscheme': 'per_channel',
#  'symmetric': True, 'group_size': None, 'layer_bits': None, 'qdq_placement': 'per_op'}

# INT8 weight-only, one scale per group of 64 weights
Quantizer(backend='torchao', weight_bits=8, act_bits=16, qscheme='per_group', group_size=64)

# Keep the classifier out of the quantization (16 = left in floating point)
Quantizer(backend='x86', weight_bits={'fc': 16})

# The same dict on torchao, over the Linear layers it rewrites: INT8 everywhere except one layer
Quantizer(backend='torchao', weight_bits={'layers.0.linear1': 16}, act_bits=16)
```

A `{layer_name: width}` dict says *which* layers are quantized; the width they are quantized *at* is the
uniform one, so the two spellings below resolve to the same cell. Layers the dict does not name keep that
uniform width — the dict is an override list, not a whitelist.

```python
_resolve_spec('torchao', weight_bits={'fc': 16}, act_bits=16).weight_bits          # 8
_resolve_spec('torchao', weight_bits={'fc': 16, 'head': 8}, act_bits=16).weight_bits  # 8
```

`qdq_placement` names where the Q/DQ pairs sit rather than how many bits they keep. Only the `pt2e`
cell has that axis, so it is the only one whose spec carries a value; everywhere else the field stays
`None` and naming a placement raises:

```python
Quantizer(backend='pt2e', qdq_placement='skip_conv_add')  # the residual conv->add edge stays unquantized
_resolve_spec('pt2e').qdq_placement                       # 'per_op' — asking for it explicitly is the same request
_resolve_spec('x86').qdq_placement                        # None — the FX flow has no such axis
```

Asking for a cell a backend cannot honor raises, and says who can:

```python
Quantizer(backend='x86', symmetric=True)
# ValueError: backend='x86' W8A8 cannot honor symmetric=True — its observers are affine by
# construction: activations carry a non-zero zero-point. The backend(s) that can: ['pt2e', 'torchao'].

Quantizer(backend='torchao', weight_bits={'fc': 8})   # W8A8: torchao's dynamic-activation recipe
# ValueError: backend='torchao' cannot honor a per-layer `weight_bits` dict at W8A8, only at W8A16:
# add act_bits=16.
```

In [ ]:
#| hide
from fastcore.test import *

# --- the matrix is well formed: every cell is reachable by its own key ---
for key, c in PRECISION_SUPPORT.items():
    test_eq(key, (c.backend, c.weight_bits, c.act_bits))
    assert c.qschemes, f"{key} names no weight axis"
    assert set(c.qschemes) <= set(QSCHEMES), f"{key} names an unknown axis"
    assert set(c.symmetries) <= {True, False}, f"{key} names an unknown symmetry"
    assert c.weight_bits in WIDTHS and c.act_bits in WIDTHS, f"{key} uses a width the grammar cannot name"
    assert c.note.endswith('.'), f"{key} has no explanation"
    # a per-group cell is the only kind allowed to carry a default group size
    if c.default_group_size is not None: assert 'per_group' in c.qschemes, key
    assert set(c.qdq_placements) <= set(QDQ_PLACEMENTS), f"{key} names an unknown Q/DQ placement"
    # a cell that places its pairs at all can place them the ordinary way
    if c.qdq_placements: assert c.default_qdq_placement == 'per_op', key

# every backend but torchao is INT8-only today: a new cell elsewhere needs its kernels checked first
assert all(c.label == 'W8A8' for c in _CELLS if c.backend != 'torchao')

# per-layer widths are a property of the CELL, not of the backend: torchao honors a dict weight-only,
# and never with the run-time activation scales of its W8A8 recipe
test_eq(sorted((c.backend, c.label) for c in _CELLS if c.per_layer),
        [('fbgemm', 'W8A8'), ('onednn', 'W8A8'), ('qnnpack', 'W8A8'), ('torchao', 'W8A16'), ('x86', 'W8A8')])

# exactly one cell exports today, and it is the symmetric pt2e one
test_eq([c.label for c in PRECISION_SUPPORT.values() if c.exports], ['W8A8'])
test_eq([c.backend for c in PRECISION_SUPPORT.values() if c.exports], ['pt2e'])

# choosing where the Q/DQ pairs sit is a property of one CELL, not of the whole grammar: pt2e is the
# only flow fasterai annotates itself, so it is the only one that can move a pair
test_eq(sorted((c.backend, c.label) for c in _CELLS if c.qdq_placements), [('pt2e', 'W8A8')])
test_eq(PRECISION_SUPPORT[('pt2e', 8, 8)].qdq_placements, QDQ_PLACEMENTS)
test_eq(QDQ_PLACEMENTS, ('per_op', 'skip_conv_add'))
for c in _CELLS:
    if not c.qdq_placements: test_eq(c.default_qdq_placement, None)

# defaults come off the front of the tuples
_pt2e = PRECISION_SUPPORT[('pt2e', 8, 8)]
test_eq((_pt2e.default_qscheme, _pt2e.default_symmetric, _pt2e.label), ('per_channel', True, 'W8A8'))
test_eq(_pt2e.default_qdq_placement, 'per_op')
test_eq(_pt2e.as_dict()['backend'], 'pt2e')
test_eq(_pt2e.as_dict()['qdq_placements'], QDQ_PLACEMENTS)
test_eq(_label(4, 16), 'W4A16')

# the rendered table shows every cell, and is generated from the matrix
_table = precision_table()
for c in PRECISION_SUPPORT.values(): assert f"`{c.backend}`" in _table and c.note in _table
test_eq(_table.count('\n'), len(PRECISION_SUPPORT) + 1)  # header + separator + one row per cell
# ...one column per thing a cell can say, header and separator agreeing (8 today: the Q/DQ placement
# column is the newest one, and a cell without that axis renders 'n/a' rather than an empty column)
_head, _sep, *_rows = _table.split('\n')
test_eq(_head.count('|'), 9)                     # 8 columns, 9 pipes
test_eq(_sep.count('|'), _head.count('|'))
for _row in _rows: test_eq(_row.count('|'), _head.count('|'))
assert '| Q/DQ placement |' in _head, _head
assert '| per_op, skip_conv_add |' in _table
assert '| n/a |' in _table

In [ ]:
#| hide
# --- resolution: the defaults every legacy call relies on ---
test_eq(_resolve_spec('x86').as_dict(),
        {'backend': 'x86', 'method': 'static', 'weight_bits': 8, 'act_bits': 8, 'qscheme': 'per_channel',
         'symmetric': False, 'group_size': None, 'layer_bits': None, 'qdq_placement': None})
test_eq(_resolve_spec('pt2e').qscheme, 'per_channel')       # per-channel weights, as pt2e has always done
test_eq(_resolve_spec('pt2e').symmetric, True)
test_eq(_resolve_spec('qnnpack').qscheme, 'per_tensor')     # qnnpack's default observer is per-tensor
test_eq(_resolve_spec('x86', 'qat').method, 'qat')          # `method` is left alone, it is the schedule axis

# a spec asked for explicitly is the same spec as the default one: the grammar is a superset
test_eq(_resolve_spec('pt2e', weight_bits=8, act_bits=8, qscheme='per_channel', symmetric=True),
        _resolve_spec('pt2e'))

# --- Q/DQ placement: an axis only the pt2e cell has ---
test_eq(_resolve_spec('pt2e').qdq_placement, 'per_op')                       # the cell's own default...
test_eq(_resolve_spec('pt2e', qdq_placement='per_op'), _resolve_spec('pt2e'))  # ...spelled out, same spec
test_eq(_resolve_spec('pt2e', qdq_placement='skip_conv_add').qdq_placement, 'skip_conv_add')
assert _resolve_spec('pt2e', qdq_placement='skip_conv_add') != _resolve_spec('pt2e')
# a backend whose flow has no such axis records nothing, the way `group_size` records nothing off
# `per_group` — the field describes the arithmetic, so it may not claim one that never ran
for _backend in ('x86', 'fbgemm', 'onednn', 'qnnpack'):
    test_eq(_resolve_spec(_backend).qdq_placement, None)
test_eq(_resolve_spec('torchao', 'int8_weight_only').qdq_placement, None)

# --- the axes have an effect ---
test_eq(_resolve_spec('pt2e', qscheme='per_tensor').qscheme, 'per_tensor')
test_eq(_resolve_spec('x86', use_per_tensor=True).qscheme, 'per_tensor')  # the legacy flag is an axis request
test_eq(_resolve_spec('torchao', weight_bits=8, act_bits=16, qscheme='per_group', group_size=64).group_size, 64)

# --- torchao: the recipe and the precision are two names for one thing ---
test_eq(_resolve_spec('torchao', 'int8_weight_only').label, 'W8A16')
test_eq(_resolve_spec('torchao', 'int8_dynamic').label, 'W8A8')
test_eq(_resolve_spec('torchao', 'int4_weight_only').as_dict()['group_size'], 128)  # its native group size
# ...so naming the precision picks the recipe
test_eq(_resolve_spec('torchao', weight_bits=4).method, 'int4_weight_only')
test_eq(_resolve_spec('torchao', weight_bits=8, act_bits=16).method, 'int8_weight_only')
test_eq(_resolve_spec('torchao', weight_bits=8, act_bits=8).method, 'int8_dynamic')
test_eq(_TORCHAO_RECIPES, {'W8A16': 'int8_weight_only', 'W8A8': 'int8_dynamic', 'W4A16': 'int4_weight_only'})

# --- per-layer widths ---
_spec = _resolve_spec('x86', weight_bits={'fc': 16})
test_eq((_spec.weight_bits, _spec.layer_bits), (8, {'fc': 16}))   # unlisted layers keep the backend default
_asked = {'fc': 16}
_spec = _resolve_spec('x86', weight_bits=_asked)
_asked['fc'] = 8
test_eq(_spec.layer_bits, {'fc': 16})  # the spec holds a copy, not the caller's dict

# --- torchao honors a per-layer dict weight-only: the width the dict does NOT name is the uniform one ---
_ao = _resolve_spec('torchao', weight_bits={'0': 16, '2': 8}, act_bits=16)
test_eq((_ao.backend, _ao.method, _ao.label), ('torchao', 'int8_weight_only', 'W8A16'))
test_eq((_ao.weight_bits, _ao.layer_bits), (8, {'0': 16, '2': 8}))
# naming the recipe instead of the precision is the same request
test_eq(_resolve_spec('torchao', 'int8_weight_only', weight_bits={'0': 16, '2': 8}), _ao)

# the scalar is DERIVED from the dict: an 8 anywhere fixes the width at 8, an all-16 dict leaves it to
# the backend (which is what keeps `weight_bits={'fc': 16}` resolving exactly as it always has)
test_eq(_split_weight_bits({'a': 16, 'b': 8}), (8, {'a': 16, 'b': 8}))
test_eq(_split_weight_bits({'a': 16, 'b': 16}), (None, {'a': 16, 'b': 16}))
test_eq(_split_weight_bits(8), (8, None))
test_eq(_split_weight_bits(None), (None, None))
test_eq(_resolve_spec('torchao', 'int8_weight_only', weight_bits={'0': 16}).weight_bits, 8)

# --- the spec is frozen, and knows what it can export ---
_spec = _resolve_spec('pt2e')
test_eq((_spec.exports, _spec.cell.backend, _spec.label), (True, 'pt2e', 'W8A8'))
test_eq(_resolve_spec('torchao', 'int8_weight_only').exports, False)
with ExceptionExpected(AttributeError): _spec.weight_bits = 4
with ExceptionExpected(AttributeError): _spec.qscheme = 'per_tensor'
assert _spec.note

# --- `quant_spec` is the one way to read the spec back off a model ---
class _Tagged: pass
_model = _Tagged()
test_eq(quant_spec(_model), None)                  # an unquantized model has no precision to report
setattr(_model, SPEC_ATTR, _spec)
test_eq(quant_spec(_model), _spec)
test_eq(SPEC_ATTR, '_fasterai_quant_spec')         # the attribute name is part of the contract

In [ ]:
#| hide
# --- every refusal is loud, names the argument, and points at a backend that can ---
def _refused(regex, *args, exc=ValueError, **kwargs):
    "Assert `_resolve_spec` refuses this request with a message matching `regex`"
    with ExceptionExpected(exc, regex=regex): _resolve_spec(*args, **kwargs)

# a precision no backend runs
_refused("No fasterai backend runs W4A8", 'pt2e', weight_bits=4, act_bits=8)
_refused("no recipe for W4A8", 'torchao', weight_bits=4, act_bits=8)
# a precision this backend does not run, but another one does
_refused("that run W8A16", 'pt2e', act_bits=16)
# an axis this backend does not offer
_refused("not 'per_channel'", 'qnnpack', qscheme='per_channel')
_refused("Unknown qscheme 'channel'", 'pt2e', qscheme='channel')
# a symmetry its observers cannot produce — the silent-drop case the FX flow used to hide
_refused("cannot honor symmetric=True", 'x86', symmetric=True)
_refused("cannot honor symmetric=True", 'fbgemm', symmetric=True)
# per-group asks and their backend
_refused("only means something with qscheme='per_group'", 'x86', group_size=64)
_refused("needs a `group_size`", 'torchao', weight_bits=8, act_bits=16, qscheme='per_group')
_refused("that do: ", 'pt2e', qscheme='per_group', group_size=64)
# a Q/DQ placement on a backend whose flow does not place its pairs — including the DEFAULT placement,
# which on such a cell is a request nothing would read rather than a harmless no-op
_refused("names no placement at all", 'x86', qdq_placement='skip_conv_add')
_refused("names no placement at all", 'x86', qdq_placement='per_op')
_refused("The backend\\(s\\) that can: \\['pt2e'\\]", 'torchao', 'int8_weight_only',
         qdq_placement='skip_conv_add')
_refused("Unknown qdq_placement 'fuse_residuals'", 'pt2e', qdq_placement='fuse_residuals')
_refused("`qdq_placement` must be one of ", 'pt2e', exc=TypeError, qdq_placement=1)
# per-layer widths on a backend that quantizes the whole graph at once
_refused("cannot honor a per-layer `weight_bits` dict", 'pt2e', weight_bits={'fc': 16})
# ...while torchao DOES honor one weight-only, so the dict resolves instead of being refused
test_eq(_resolve_spec('torchao', 'int8_weight_only', weight_bits={'fc': 16}).layer_bits, {'fc': 16})
# ...and its other cell, whose activation scales are computed at run time, names the way to the one
# that can rather than sending the caller to another backend
_refused("only at W8A16: add act_bits=16", 'torchao', weight_bits={'fc': 8})
_refused("only at W8A16: add act_bits=16", 'torchao', 'int8_dynamic', weight_bits={'fc': 8})
_refused("only at W8A16: add act_bits=16", 'torchao', weight_bits={'fc': 16}, act_bits=8)
# a per-layer width is 8-or-16 on every backend: there is no per-layer INT4 in this alphabet
_refused("a per-layer width is 8", 'torchao', weight_bits={'fc': 4}, act_bits=16)
# contradictions between two ways of asking for the same thing
_refused("ask for different weight axes", 'x86', use_per_tensor=True, qscheme='per_channel')
_refused("never read", 'pt2e', use_per_tensor=True)
_refused("contradicts", 'torchao', 'int8_weight_only', act_bits=8)
_refused("has no method 'invalid'", 'torchao', 'invalid')

# --- plausible-wrong values get a clear error, never a cryptic one downstream ---
# every type complaint reads the same: "`name` must be ..., got value (type)."
_refused("`weight_bits` must be an int", 'pt2e', exc=TypeError, weight_bits='8')
_refused("`act_bits` must be an int", 'pt2e', exc=TypeError, act_bits=8.0)
_refused("is not a width this grammar names", 'pt2e', act_bits=32)
_refused("is not a width this grammar names", 'pt2e', weight_bits=2)
_refused("`symmetric` must be True, False or None", 'pt2e', exc=TypeError, symmetric='yes')
_refused("`qscheme` must be one of ", 'pt2e', exc=TypeError, qscheme=8)
_refused("`group_size` must be a positive int", 'torchao', weight_bits=4, exc=TypeError, group_size='128')
_refused("is not a size", 'torchao', weight_bits=4, group_size=0)
_refused("names no layer", 'x86', weight_bits={})
_refused("`weight_bits keys` must be layer names", 'x86', exc=TypeError, weight_bits={1: 8})
_refused("a per-layer width is 8", 'x86', weight_bits={'fc': 4})
_refused("Unknown backend 'x87'", 'x87')
_refused("`backend` must be a str", exc=TypeError, backend=8)

# --- an unportable cell is allowed, but it warns ---
import warnings
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    test_eq(_resolve_spec('pt2e', symmetric=False).symmetric, False)
assert any('zero-points' in str(w.message) for w in _caught), [str(w.message) for w in _caught]

In [ ]:
#| hide
# --- the matrix says what torch actually does (this is what keeps it from drifting) ---
import torch
from torch.ao.quantization import get_default_qconfig

_TORCH_AXIS = {torch.per_channel_symmetric: 'per_channel', torch.per_channel_affine: 'per_channel',
               torch.per_tensor_symmetric: 'per_tensor', torch.per_tensor_affine: 'per_tensor'}

for _backend in ('x86', 'fbgemm', 'onednn', 'qnnpack'):
    _qconfig = get_default_qconfig(_backend)
    _cell = PRECISION_SUPPORT[(_backend, 8, 8)]
    # the default weight axis of the matrix is the one the default observer uses
    test_eq(_TORCH_AXIS[_qconfig.weight().qscheme], _cell.default_qscheme)
    # ...and the activations are affine, which is why these cells cannot claim symmetric=True
    test_eq(_qconfig.activation().qscheme, torch.per_tensor_affine)
    test_eq(_cell.symmetries, (False,))
    assert not _cell.exports

---

## See Also

- [Granularity](granularity.html) - What block of parameters to remove
- [Criteria](criteria.html) - Which parameters matter
- [Schedules](schedules.html) - When compression happens
- [Quantizer](../quantize/quantizer.html) - The class that applies a precision cell
- [ONNX Exporter](../export/onnx_exporter.html) - `export_qdq`, which refuses the cells it cannot write